In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

In [ ]:
# Load and preprocess the MNIST dataset

(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train, x_test = x_train / 255.0, x_test / 255.0

x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

(60000, 784) (60000, 10) (10000, 784) (10000, 10)


In [ ]:
def create_model():
    inputs = Input(shape=(784,))
    x = Dense(2, activation='relu')(inputs)
    x = Dense(4, activation='relu')(x)
    x = Dense(2, activation='relu')(x)
    outputs = Dense(10, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

tape_model = create_model()

In [ ]:
loss_fn = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

In [ ]:
# training and validation splits
train_test_split = int(0.85 * len(x_train))
trainX = x_train[0:train_test_split]
trainY = y_train[0:train_test_split]
x_val = x_train[train_test_split:]
y_val = y_train[train_test_split:]

In [ ]:
# convert TensorFlow Datasets for training and validation
train_ds = tf.data.Dataset.from_tensor_slices((trainX, trainY))
train_ds = train_ds.shuffle(buffer_size=10000).batch(128)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_ds = val_ds.batch(128)

In [ ]:
for epoch in range(5):
    # Training loop
    for x_batch, y_batch in train_ds:
        with tf.GradientTape() as tape:
            probs = tape_model(x_batch)
            loss = loss_fn(y_batch, probs)
        grads = tape.gradient(loss, tape_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, tape_model.trainable_variables))

    val_losses = []
    val_accuracy = tf.keras.metrics.CategoricalAccuracy()

    for x_val_batch, y_val_batch in val_ds:
        val_probs = tape_model(x_val_batch)
        val_loss = loss_fn(y_val_batch, val_probs)
        val_losses.append(val_loss.numpy())
        val_accuracy.update_state(y_val_batch, val_probs)

    val_loss_avg = sum(val_losses) / len(val_losses)

    print(f"Epoch {epoch+1}: validation accuracy = {val_accuracy.result().numpy():.4f}")

Epoch 1: validation accuracy = 0.3832
Epoch 2: validation accuracy = 0.3919
Epoch 3: validation accuracy = 0.4774
Epoch 4: validation accuracy = 0.4773
Epoch 5: validation accuracy = 0.4473


In [ ]:
# compile the model for Keras .fit() training
fit_model = create_model()
fit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

fit_model.fit(x_train, y_train, batch_size=128, epochs=5, validation_split=0.2, shuffle=False)

def evaluate(model, x_test_data, y_test_labels):
    y_pred_prob = model(x_test_data)
    y_pred = tf.argmax(y_pred_prob, axis=1)
    y_true = tf.argmax(y_test_labels, axis=1)
    accuracy = tf.reduce_mean(tf.cast(tf.equal(y_pred, y_true), tf.float32))
    return accuracy

Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.1951 - loss: 2.0186 - val_accuracy: 0.2619 - val_loss: 1.7771
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2569 - loss: 1.7599 - val_accuracy: 0.2646 - val_loss: 1.7372
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.2692 - loss: 1.7361 - val_accuracy: 0.2779 - val_loss: 1.7253
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.2786 - loss: 1.7252 - val_accuracy: 0.2769 - val_loss: 1.7174
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2864 - loss: 1.7143 - val_accuracy: 0.2621 - val_loss: 1.7036


In [ ]:
print(f"\nGradient Tape accuracy: {evaluate(tape_model, x_test, y_test):.2f}")
print(f"Using Keras fit accuracy: {evaluate(fit_model, x_test, y_test):.2f}")


Gradient Tape accuracy: 0.44
Using Keras fit accuracy: 0.26
